# Variant C (Transformer) Sub-100M on T4

Short-run setup for Kaggle T4 testing with checkpoint resume support.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys

REQUIRED_PIP_PACKAGES = {
    'miditok': 'miditok',
    'pretty_midi': 'pretty_midi',
    'ncps': 'ncps',
    'safetensors': 'safetensors',
    'matplotlib': 'matplotlib',
    'huggingface_hub': 'huggingface_hub',
}

def ensure_runtime_dependencies() -> None:
    missing_specs = [
        spec
        for module_name, spec in REQUIRED_PIP_PACKAGES.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if not missing_specs:
        print('Runtime dependencies ready.')
        return

    print(f'Installing missing package(s): {missing_specs}')
    try:
        subprocess.check_call(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--disable-pip-version-check',
                *missing_specs,
            ]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to install required packages. On Kaggle, enable Internet access or attach a dataset containing wheels.'
        ) from exc

    importlib.invalidate_caches()
    still_missing = [
        module_name
        for module_name in REQUIRED_PIP_PACKAGES
        if importlib.util.find_spec(module_name) is None
    ]
    if still_missing:
        raise ModuleNotFoundError(f'Required packages still missing after install: {still_missing}')
    print('Runtime dependencies installed.')

ensure_runtime_dependencies()

TRAIN_MODULE = 'training.train_variant_c_sub100m'
TRAIN_SCRIPT = Path('training/train_variant_c_sub100m.py')

def can_write_to(path: Path) -> bool:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / '.copilot_write_probe'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    probes = [start]
    if start.name.lower() == 'notebooks':
        probes.append(start.parent)

    kaggle_hints = [
        Path('/kaggle/working'),
        Path('/kaggle/input'),
        Path('/kaggle/working/piano_midi_model'),
        Path('/kaggle/input/piano_midi_model'),
        Path('/kaggle/input/piano-midi-model'),
        Path('/kaggle/input/midi-ai-piano-model'),
    ]
    probes.extend(path for path in kaggle_hints if path.exists())

    seen = set()
    for probe in probes:
        if not probe.exists():
            continue
        key = str(probe)
        if key in seen:
            continue
        seen.add(key)
        for parent in [probe, *probe.parents]:
            if (parent / TRAIN_SCRIPT).exists() and (parent / 'training' / '__init__.py').exists():
                return parent

    for scan_root in (Path('/kaggle/working'), Path('/kaggle/input')):
        if not scan_root.exists():
            continue
        hit = next(scan_root.rglob(str(TRAIN_SCRIPT).replace('\\', '/')), None)
        if hit is None:
            continue
        candidate = hit.parent.parent
        if (candidate / 'training' / '__init__.py').exists():
            return candidate

    raise FileNotFoundError(
        f'Could not locate project root containing {TRAIN_SCRIPT}. '
        'Copy the full repository into Kaggle input/working and retry.'
    )

def resolve_output_base(project_root: Path) -> Path:
    candidates = []
    kaggle_working = Path('/kaggle/working')
    if kaggle_working.exists():
        candidates.append(kaggle_working)
    candidates.append(project_root)

    for candidate in candidates:
        if can_write_to(candidate):
            return candidate

    raise OSError(
        'Could not find a writable output base. On Kaggle, keep code in /kaggle/input and write to /kaggle/working.'
    )

PROJECT_ROOT = find_project_root()
OUTPUT_BASE = resolve_output_base(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')

print(f'Project root: {PROJECT_ROOT}')
print(f'Output base: {OUTPUT_BASE}')
print(f'Training module: {TRAIN_MODULE}')

In [ ]:
import numpy as np
import torch

OUTPUT_DIR = OUTPUT_BASE / 'outputs' / 'variant_c_80m_t4'
SIZE_PRESET = '80m'  # '40m' or '80m'
MAX_PIECES = 15000
EPOCHS = 6
TARGET_EFFECTIVE_BATCH = 16
BATCH_PER_GPU = 2  # drop to 1 if you hit OOM on dual T4
LEARNING_RATE = 2e-4
SEED = 42
LOG_EVERY_N_STEPS = 20
RESUME_MODE = 'remaining'  # 'remaining' or 'additional'
AUTO_RESUME = True
SEED_MIDI = ''  # optional

GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 1
DATA_PARALLEL_ACTIVE = GPU_COUNT > 1
if DATA_PARALLEL_ACTIVE:
    BATCH_SIZE = max(GPU_COUNT, GPU_COUNT * BATCH_PER_GPU)
else:
    BATCH_SIZE = 1
GRAD_ACCUM_STEPS = max(1, int(np.ceil(float(TARGET_EFFECTIVE_BATCH) / float(BATCH_SIZE))))
NUM_WORKERS = max(2, int(GPU_COUNT) * 2)

FORCE_REBUILD_MANIFEST = True
PRETOKENIZED_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'processed',
    Path('/kaggle/input/godzilla-tokenized-15k/tokenized'),
    Path('/kaggle/input/godzilla-tokenized-15k'),
    Path('/kaggle/input/godzilla-piano-tokenized/tokenized'),
    Path('/kaggle/input/godzilla-piano-tokenized'),
]

def find_resume_checkpoint(checkpoint_dir: Path) -> str:
    candidates = [
        checkpoint_dir / 'latest_state.pt',
        checkpoint_dir / 'latest.safetensors',
        checkpoint_dir / 'best_state.pt',
        checkpoint_dir / 'best.safetensors',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return ''

def discover_npz_parent(path: Path):
    if not path.exists():
        return None
    hit = next(path.rglob('*.npz'), None)
    if hit is None:
        return None
    return hit.parent

def find_npz_root(candidates) -> Path:
    for candidate in candidates:
        resolved = discover_npz_parent(Path(candidate))
        if resolved is not None:
            return resolved

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for folder in kaggle_input.iterdir():
            if not folder.is_dir():
                continue
            resolved = discover_npz_parent(folder)
            if resolved is not None:
                return resolved

    raise FileNotFoundError(
        'Could not locate tokenized NPZ files. Set PRETOKENIZED_ROOT_CANDIDATES to your dataset path.'
    )

def _resolve_manifest_npz_path(raw_path: str, *, manifest_path: Path, pretokenized_root: Path | None):
    candidate = Path(str(raw_path))
    probe_paths = []
    if candidate.is_absolute():
        probe_paths.append(candidate)
    else:
        if pretokenized_root is not None:
            probe_paths.append(pretokenized_root / candidate)
            probe_paths.append(pretokenized_root / candidate.name)
        probe_paths.append(manifest_path.parent / candidate)
        probe_paths.append(manifest_path.parent.parent / candidate)
        probe_paths.append(manifest_path.parent.parent / 'data' / candidate.name)
    for path in probe_paths:
        if path.exists() and path.is_file():
            return path
    return None

def manifest_has_resolvable_entries(manifest_path: Path, pretokenized_root: Path | None) -> bool:
    if not manifest_path.exists():
        return False
    try:
        raw = json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception:
        return False
    if not isinstance(raw, list) or not raw:
        return False
    resolved = 0
    for row in raw:
        if not isinstance(row, dict):
            continue
        raw_npz = str(row.get('npz_path', '')).strip()
        if not raw_npz:
            md5 = str(row.get('md5', '')).strip()
            if md5:
                raw_npz = f'{md5}.npz'
        if not raw_npz:
            continue
        if _resolve_manifest_npz_path(raw_npz, manifest_path=manifest_path, pretokenized_root=pretokenized_root) is not None:
            resolved += 1
            if resolved >= 2:
                return True
    return False

def build_manifest_from_npz(npz_root: Path, manifest_path: Path) -> int:
    npz_files = sorted(npz_root.rglob('*.npz'))
    if not npz_files:
        raise FileNotFoundError(f'No .npz files found under: {npz_root}')

    manifest = []
    skipped = 0
    for npz_path in npz_files:
        try:
            with np.load(npz_path, allow_pickle=False) as pack:
                if 'tokens' not in pack.files:
                    skipped += 1
                    continue
                token_len = int(np.asarray(pack['tokens']).shape[0])
        except Exception:
            skipped += 1
            continue

        manifest.append({
            'md5': npz_path.stem,
            'npz_path': str(npz_path),
            'length': token_len,
            'source_path': str(npz_path.parent),
        })

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f'Manifest built: {manifest_path} | entries={len(manifest)} | skipped={skipped}')
    return len(manifest)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRETOKENIZED_ROOT = find_npz_root(PRETOKENIZED_ROOT_CANDIDATES)
PRETOKENIZED_MANIFEST = OUTPUT_DIR / 'processed_pretokenized' / 'manifest.json'
needs_rebuild = FORCE_REBUILD_MANIFEST or not manifest_has_resolvable_entries(PRETOKENIZED_MANIFEST, PRETOKENIZED_ROOT)
if needs_rebuild:
    _ = build_manifest_from_npz(PRETOKENIZED_ROOT, PRETOKENIZED_MANIFEST)
else:
    print(f'Using existing manifest: {PRETOKENIZED_MANIFEST}')

checkpoint_dir = OUTPUT_DIR / 'checkpoints' / f'variant_c_{SIZE_PRESET}'
resume_ckpt = find_resume_checkpoint(checkpoint_dir) if AUTO_RESUME else ''
RESUME_FROM_CHECKPOINT = resume_ckpt if resume_ckpt else ''
print(f'Pretokenized root: {PRETOKENIZED_ROOT}')
print(f'Manifest: {PRETOKENIZED_MANIFEST}')
print(f'Output base: {OUTPUT_BASE}')
print(f'Output: {OUTPUT_DIR}')
print(f'GPU count: {GPU_COUNT} | DataParallel target: {DATA_PARALLEL_ACTIVE}')
print(f'Batch size: {BATCH_SIZE} | Grad accumulation: {GRAD_ACCUM_STEPS} | Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'Num workers: {NUM_WORKERS}')
print(f'Resume checkpoint: {resume_ckpt or "none"}')

In [ ]:
cmd = [
    sys.executable, '-m', TRAIN_MODULE,
    '--size_preset', SIZE_PRESET,
    '--pretokenized_manifest', str(PRETOKENIZED_MANIFEST),
    '--pretokenized_root', str(PRETOKENIZED_ROOT),
    '--output_dir', str(OUTPUT_DIR),
    '--max_pieces', str(MAX_PIECES),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--grad_accumulation_steps', str(GRAD_ACCUM_STEPS),
    '--num_workers', str(NUM_WORKERS),
    '--learning_rate', str(LEARNING_RATE),
    '--seed', str(SEED),
    '--log_every_n_steps', str(LOG_EVERY_N_STEPS),
]
if RESUME_FROM_CHECKPOINT:
    cmd.extend(['--resume_from_checkpoint', RESUME_FROM_CHECKPOINT, '--resume_mode', RESUME_MODE])
if str(SEED_MIDI).strip():
    cmd.extend(['--seed_midi', str(SEED_MIDI)])
print(shlex.join(cmd))

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    if not PRETOKENIZED_MANIFEST.exists():
        raise FileNotFoundError(f'Manifest not found: {PRETOKENIZED_MANIFEST}')
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + run_env.get('PYTHONPATH', '')
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=run_env, check=True)
else:
    print('Dry run only. Set RUN_TRAINING=True to launch.')

result_path = OUTPUT_DIR / f'variant_c_{SIZE_PRESET}_result.json'
if result_path.exists():
    payload = json.loads(result_path.read_text(encoding='utf-8'))
    print('Result path:', result_path)
    print('Params:', payload.get('result', {}).get('params'))
    val_loss = payload.get('result', {}).get('val_loss', [])
    if val_loss:
        print('Last val loss:', val_loss[-1])
    print('Resume metadata:', payload.get('result', {}).get('resume', {}))
    print('Backend status:', payload.get('backend_status', {}))
else:
    print('No result JSON yet.')

In [ ]:
from pathlib import Path
import torch
from safetensors.torch import load_file as safetensors_load_file

from config import DataConfig, ModelConfig
from data.tokenizer import PianoTokenizer
from data.tokenizer_custom import CustomDeltaTokenizer
from generation.generate import GenerationConfig, generate_continuation
from model.factory import build_model
from utils.config_compat import normalize_model_config_payload

VARIANT_PREFIX = 'variant_c'

def find_blue_bird_seed() -> Path:
    direct_candidates = [
        PROJECT_ROOT / 'bluebird.mid',
        PROJECT_ROOT / 'Blue Bird.mid',
        PROJECT_ROOT / 'BlueBird.mid',
        PROJECT_ROOT / 'blue bird.mid',
        OUTPUT_BASE / 'bluebird.mid',
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    search_roots = [PROJECT_ROOT, OUTPUT_BASE, Path('/kaggle/input')]
    for root in search_roots:
        if not root.exists():
            continue
        for midi_path in root.rglob('*.mid'):
            if 'generated' in {part.lower() for part in midi_path.parts}:
                continue
            normalized = midi_path.name.lower().replace(' ', '')
            if 'bluebird' in normalized:
                return midi_path
        for midi_path in root.rglob('*.midi'):
            if 'generated' in {part.lower() for part in midi_path.parts}:
                continue
            normalized = midi_path.name.lower().replace(' ', '')
            if 'bluebird' in normalized:
                return midi_path

    raise FileNotFoundError(
        'Could not find Blue Bird MIDI under the project or dataset root.'
    )

def resolve_checkpoint_bundle(checkpoint_dir: Path):
    state_candidates = [
        checkpoint_dir / 'latest_state.pt',
        checkpoint_dir / 'best_state.pt',
    ]
    state_path = next((path for path in state_candidates if path.exists()), None)
    if state_path is None:
        raise FileNotFoundError(f'No checkpoint state found in {checkpoint_dir}')

    state = torch.load(state_path, map_location='cpu')
    weights_name = str(state.get('model_weights_path', '')).strip()
    model_candidates = []
    if weights_name:
        model_candidates.append(checkpoint_dir / weights_name)
    model_candidates.extend([
        checkpoint_dir / 'latest.safetensors',
        checkpoint_dir / 'best.safetensors',
    ])
    model_path = next((path for path in model_candidates if path.exists()), None)
    if model_path is None:
        raise FileNotFoundError(
            f'No model weights found next to checkpoint state {state_path}'
        )
    return state_path, model_path, state

def load_tokenizer_for_generation(tokenizer_path: Path, tokenization_strategy: str):
    strategy = str(tokenization_strategy).strip().lower()
    if strategy == 'custom_delta':
        return CustomDeltaTokenizer.load(str(tokenizer_path))
    return PianoTokenizer.load(str(tokenizer_path))

checkpoint_dir = OUTPUT_DIR / 'checkpoints' / f'{VARIANT_PREFIX}_{SIZE_PRESET}'
state_path, model_path, checkpoint_state = resolve_checkpoint_bundle(checkpoint_dir)

data_cfg_payload = dict(checkpoint_state.get('data_config') or {})
data_cfg = DataConfig(**data_cfg_payload)
tokenizer_candidates = [
    Path(str(data_cfg.tokenizer_path)).expanduser(),
    OUTPUT_DIR / 'custom_tokenizer.json',
    checkpoint_dir.parent.parent / 'custom_tokenizer.json',
    checkpoint_dir.parent / 'custom_tokenizer.json',
]
tokenizer_path = next((path for path in tokenizer_candidates if path.exists()), None)
if tokenizer_path is None:
    raise FileNotFoundError(
        f'No tokenizer found for generation. Looked under {OUTPUT_DIR} and {checkpoint_dir}.'
    )

tokenizer = load_tokenizer_for_generation(
    tokenizer_path=tokenizer_path,
    tokenization_strategy=str(getattr(data_cfg, 'tokenization_strategy', '')),
)
model_cfg = ModelConfig(
    **normalize_model_config_payload(dict(checkpoint_state.get('model_config') or {}))
)
model_cfg.vocab_size = tokenizer.vocab_size
data_cfg.vocab_size = tokenizer.vocab_size
data_cfg.tokenizer_path = str(tokenizer_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU for generation.')

model = build_model(model_cfg)
model_state = safetensors_load_file(str(model_path), device='cpu')
missing, unexpected = model.load_state_dict(model_state, strict=False)
if missing:
    print(f'Missing keys while loading checkpoint: {len(missing)}')
if unexpected:
    print(f'Unexpected keys while loading checkpoint: {len(unexpected)}')
model.to(device)
model.eval()

seed_path = find_blue_bird_seed()
generated_dir = OUTPUT_DIR / 'generated'
generated_dir.mkdir(parents=True, exist_ok=True)
output_path = generated_dir / f'bluebird_{VARIANT_PREFIX}_{SIZE_PRESET}_continuation.mid'

PARALLEL_GENERATION_SAMPLES = 2 if device.type == 'cuda' and torch.cuda.device_count() > 1 else 1
generation_config = GenerationConfig(
    max_new_tokens=8192 if device.type == 'cuda' else 4096,
    temperature=0.9,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
    repetition_window=64,
    min_tokens_to_keep=3,
    num_samples=PARALLEL_GENERATION_SAMPLES,
)

print(f'Seed MIDI: {seed_path}')
print(f'Checkpoint state: {state_path}')
print(f'Checkpoint weights: {model_path}')
print(f'Output MIDI: {output_path}')
print(f'Parallel samples: {generation_config.num_samples}')
print(f'Max new tokens: {generation_config.max_new_tokens}')

outputs = generate_continuation(
    model=model,
    tokenizer=tokenizer,
    seed_midi_path=seed_path,
    output_path=output_path,
    config=data_cfg,
    generation_config=generation_config,
)

if len(outputs) == 1:
    print(f'Generated continuation: {outputs[0]}')
else:
    print('Generated continuations:')
    for output in outputs:
        print(f'  {output}')